# DANDI 000004: AMulti-Database Demo

## Overview: Neuronal Electrophysiology Analysis Using [DANDI 000004](https://dandi.github.io/example-notebooks/#dandiset-000004)

Neuroscience is a vast field, subject to domain experts. As such, there are terabytes of publicly available datasets on the topic, running from literature review to experimental design and analysis. One such database containing these datasets is DANDI: Distributed Archives for Neurophysiology Data Integration. Considered the gold standard for electrophysiology, DANDI uses the .nwb format (Neurodata Without Borders, a proprietary data format, organized into “Dandisets”) that plays nicely with vector databases, graph databases, and tabular databases. We explain our proposed approach below, using Dandiset 000004, “A NWB-based dataset and processing pipeline of human single-neuron activity during a declarative memory task”.

More on the DANDI API: https://dandi.readthedocs.io/en/latest/modref/dandiapi.html

This notebook demonstrates querying data that was ingested from a single streamed NWB session
(DANDI dandiset 000004) into the following databases:

| Database | Type | Purpose |
|---|---|---|
| **Neo4j** | Graph | Brain hierarchy (Subject -> Session -> Neuron -> Region) |
| **PostgreSQL** | Tabular | Subjects, sessions, neurons, trials |

In [1]:
import os
from dotenv import load_dotenv

# sanity checks
load_dotenv()
# print(os.environ.get("PG_DSN"))
# print(os.environ.get("NEO4J_URI"))

True

In [2]:
# create tables from schema.sql
import psycopg, os
from dotenv import load_dotenv
load_dotenv()

schema_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'experiments', 'database', 'schema.sql')
schema_path = 'schema.sql'

with open(schema_path) as f:
    sql = f.read()

with psycopg.connect(os.environ['PG_DSN']) as conn, conn.cursor() as cur:
    cur.execute(sql)
    conn.commit()

In [3]:
# Stream and ingest every NWB session from DANDI 000004 into Postgres + Neo4j.
# No files are downloaded — data is fetched on-demand via HTTP range requests.
# Already-ingested sessions are detected from the DB and skipped automatically.
%run -i "data/ingest.py"

Dandiset 000004: 87 NWB assets found, 0 already ingested.


--- sub-P10HMH/sub-P10HMH_ses-20060901_ecephys+image.nwb ---
Session: H10_7
[Postgres] Ingested H10_7: 38 neurons, 200 trials.
[Neo4j]   Ingested H10_7: 38 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-7studd_ecephys+image.nwb ---
Session: H14_18
[Postgres] Ingested H14_18: 33 neurons, 150 trials.
[Neo4j]   Ingested H14_18: 33 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-a5xz9n_ecephys+image.nwb ---
Session: H15_21
[Postgres] Ingested H15_21: 14 neurons, 150 trials.
[Neo4j]   Ingested H15_21: 14 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-1t8wrd5_ecephys+image.nwb ---
Session: H14_17
[Postgres] Ingested H14_17: 23 neurons, 200 trials.
[Neo4j]   Ingested H14_17: 23 neurons.

--- sub-P11HMH/sub-P11HMH_ses-20061101_ecephys+image.nwb ---
Session: H11_9
[Postgres] Ingested H11_9: 25 neurons, 200 trials.
[Neo4j]   Ingested H11_9: 25 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-bdd49u_ecephys+image.nwb

In [4]:
import sys
import os
import pandas as pd
from pynwb import NWBHDF5IO

# make sure to change this path to the correct NWB file
# usually when you change NWB_URL and NWB_PATH in ingest.py,
# the NWB file will be downloaded to the same path
nwb_file_path = "data/sub-P11HMH_ses-20061101_ecephys+image.nwb"

# Open the NWB file in read mode
io = NWBHDF5IO(nwb_file_path, 'r')
nwbfile = io.read()
display(nwbfile)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/sub-P11HMH_ses-20061101_ecephys+image.nwb', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## Neo4j: Graph Queries

Node breakdown:

```
(Subject)-[:HAS_SESSION]->(Session)-[:HAS_NEURON]->(Neuron)-[:LOCATED_IN]->(BrainRegion)
(Session)-[:PRESENTED]->(Stimulus)
```

In [29]:
import importlib
import utils.neo4j
importlib.reload(utils.neo4j)
from utils.neo4j import *

In [34]:
# graph node counts
summary = get_graph_summary()
print('Graph node counts:')
display(summary)

Graph node counts:


,labels,count
0,[Neuron],1839
1,[Session],87
2,[Subject],59
3,[BrainRegion],4


In [31]:
# list available brain regions
brain_regions = get_brain_regions()
print('Brain regions:')
display(brain_regions)

Brain regions:


,brain_region,n_neurons
0,Right Amygdala,577
1,Left Amygdala,494
2,Right Hippocampus,403
3,Left Hippocampus,365


In [32]:
# neurons_per_session
neurons_per_session = get_neurons_per_session()
print('Neurons per session:')
display(neurons_per_session)

Neurons per session:


,session_id,n_neurons
0,CS29_69,64
1,CS33_76,53
2,H27_58,49
3,CS54_125,46
4,H28_48,45
...,...,...
82,T88_2002,4
83,T103_2009,3
84,T107_2007,3
85,CS58_133,2


## PostgreSQL: Neuron Firing Statistics

After getting the data from Neo4j, we can refine our queries (and questions) even more with additional Postgres transactions.

In [ ]:
# if you update any function in postgres.py
# re-run this cell to get the lastest version
from utils.postgres import *

In [ ]:
# session overview
sessions = get_session_summary()
print(f'Ingested sessions: {len(sessions)}')
display(sessions)

In [ ]:
# firing statistics for ALL regions ALL sessions
stats = firing_stats()
print('Firing statistics by brain region (all sessions):')
print(stats)

In [ ]:
# filter: firing stats only for HIPPOCAMPAL neurons
hipp_stats = firing_stats_hippocampus()
print('Firing statistics for Hippocampus neurons (all sessions):')
print(stats)